In [1]:
import numpy as np
import pandas as pd
import snowflake.connector
from datetime import datetime, timedelta
import requests
import io
from scipy.optimize import linprog
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, value

In [2]:
df_plan_compra_general = pd.read_csv(r"C:\Users\juane\Downloads\PLAN_COMPRA_GENERAL_TEST.csv")

In [3]:
df_plan_compra_general["WAREHOUSE_PRODUCT_ID"]=df_plan_compra_general["Warehouseid"].astype(str)+"-"+df_plan_compra_general["Sku Id"].astype(str)

df_plan_compra_general["ADJUSTED_QUANTITY"] = df_plan_compra_general["Purchase Quantity"]

df_plan_compra_general["Variable"] = df_plan_compra_general.apply(lambda row: LpVariable(f"Q_{row['WAREHOUSE_PRODUCT_ID']}", lowBound=row["ADJUSTED_QUANTITY"]), axis=1)

filtered_df = df_plan_compra_general[df_plan_compra_general["Primary Supplier Id"] == 4]

model = LpProblem("Optimization_Plan", LpMinimize)

# Función objetivo: minimizar los días de inventario (purchase_quantity / average forecast for next 7 days)
model += lpSum([var * (1 / avg_forecast) for var, avg_forecast in zip(filtered_df["Variable"], filtered_df["Average Forecasts for Next 7 Days"])]), "Minimize_Increments"
# Restricción: suma de las cantidades igual a 250
model += lpSum(filtered_df["Variable"]) == 250, "Sum_Adjusted_Quantity"
#Restricción: El aumento del valor de cada variable no puede ser mayor a dos veces el valor inicial
for index, row in filtered_df.iterrows():
    variable = row["Variable"]
    initial_value = row["ADJUSTED_QUANTITY"]
    model += variable <= initial_value * 2, f"Max_Increase_{row['WAREHOUSE_PRODUCT_ID']}"
# Resolver el modelo
model.solve()
# Actualizar la columna "ADJUSTED_QUANTITY" con los valores optimizados
df_plan_compra_general.loc[df_plan_compra_general["Primary Supplier Id"] == 4, "ADJUSTED_QUANTITY"] = (filtered_df["Variable"].apply(lambda var: var.varValue))

# Mostrar resultados actualizados
filtered_df = df_plan_compra_general[df_plan_compra_general["Primary Supplier Id"] == 4]

print(filtered_df[["WAREHOUSE_PRODUCT_ID","Purchase Quantity", "ADJUSTED_QUANTITY"]])

   WAREHOUSE_PRODUCT_ID  Purchase Quantity  ADJUSTED_QUANTITY
3               17-4111                 20                 20
4               67-4111                 16                 16
5               47-4111                 14                 14
6                9-4111                  9                  9
7               14-4111                  9                  9
8               50-4111                  9                169
9               11-4111                  7                  7
12               8-4111                  3                  3
13              19-4111                  2                  2
14              24-4111                  1                  1
15             110-4111                  0                  0
